In [12]:
# before running, please install python, pandas and statsmodels first
import zipfile
import pickle
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [13]:
# unzip and load data frame
with zipfile.ZipFile('../Data/preprocessed_data.zip', 'r') as zipf:
    name = zipf.namelist()[0] 
    with zipf.open(name) as file:
        df = pickle.load(file)

In [14]:
# define logit model function (exogenous regime-switching logit)
def regime_switching_logit(df = pd.DataFrame):
    results = []

    # separate logit models for suspected withholding and suspected push-in
    df_mod1 = df.loc[(df['competitive_dispatch_status_discretized']== 1)]
    df_mod2 = df.loc[(df['competitive_dispatch_status_discretized']== 0)]

    X1 = df_mod1[['net_profit_withhold_marginal']]
    X1 = sm.add_constant(X1)

    y1 = (df_mod1['deviation_discretized'] == -1 ).astype(int) # values -1 becomes 1, otherwise 0

    model1 = sm.Logit(y1, X1)
    groups1 = df_mod1.index.get_level_values('EIC') # clustering standard errors by generation unit
    result1 = model1.fit(
        maxiter=1000,
        method='nm',
        cov_type='cluster',
        cov_kwds={'groups': groups1}
    )

    X2 = df_mod2[['net_profit_pushin_marginal']]
    X2 = sm.add_constant(X2)

    y2 = (df_mod2['deviation_discretized'] == 1 ).astype(int) # keep values 1, otherwise 0

    model2 = sm.Logit(y2, X2)
    groups2 = df_mod2.index.get_level_values('EIC')
    result2 = model2.fit(
        maxiter=1000,
        method='nm',
        cov_type='cluster',
        cov_kwds={'groups': groups2}
    )

    results.append(result1)
    results.append(result2)

    return results

# same model without clustering standard errors, for model runs of single units in sample
def regime_switching_logit_single_unit(df = pd.DataFrame):
    results = []

    # separate logit models for suspected withholding and suspected push-in
    df_mod1 = df.loc[(df['competitive_dispatch_status_discretized']== 1)]
    df_mod2 = df.loc[(df['competitive_dispatch_status_discretized']== 0)]

    X1 = df_mod1[['net_profit_withhold_marginal']]
    X1 = sm.add_constant(X1)

    y1 = (df_mod1['deviation_discretized'] == -1 ).astype(int) # values -1 becomes 1, otherwise 0

    model1 = sm.Logit(y1, X1)
    result1 = model1.fit(
        maxiter=1000,
        method='nm'
    )

    X2 = df_mod2[['net_profit_pushin_marginal']]
    X2 = sm.add_constant(X2)

    y2 = (df_mod2['deviation_discretized'] == 1 ).astype(int) # keep values 1, otherwise 0

    model2 = sm.Logit(y2, X2)
    result2 = model2.fit(
        maxiter=1000,
        method='nm'
    )

    results.append(result1)
    results.append(result2)

    return results

## Run model: main specification, market-level

In [15]:
results = regime_switching_logit(df)

for i, result in enumerate(results):
    print(f"Model {i+1} Summary:\n")
    print(result.summary())

Optimization terminated successfully.
         Current function value: 0.476884
         Iterations: 62
         Function evaluations: 120
Optimization terminated successfully.
         Current function value: 0.557106
         Iterations: 67
         Function evaluations: 125
Model 1 Summary:

                             Logit Regression Results                            
Dep. Variable:     deviation_discretized   No. Observations:               727741
Model:                             Logit   Df Residuals:                   727739
Method:                              MLE   Df Model:                            1
Date:                   Sun, 15 Feb 2026   Pseudo R-squ.:                 0.03128
Time:                           19:37:33   Log-Likelihood:            -3.4705e+05
converged:                          True   LL-Null:                   -3.5825e+05
Covariance Type:                 cluster   LLR p-value:                     0.000
                                   coef    std e

## Run model: slice by company

In [16]:
# rank companies and run model in order
comp_size = pd.DataFrame(df.groupby('company')['net_exposure'].agg(['min', 'max'])) #determine company size based on max. net exposure
comp_size['abs'] = comp_size[['min', 'max']].abs().max(axis=1)
comp_size = comp_size.sort_values('abs', ascending = False)
comp_size['rank'] = range(1, len(comp_size)+1) #rank companies by size
comp_size = comp_size.reset_index()
rank_df = pd.DataFrame({'company': comp_size['company'], 'rank': comp_size['rank']})

# run function for each company in order of size
results = []
for comp in rank_df['company']:
    df_comp = df.loc[df['company'] == comp]
    print(f'{comp}')
    try:
        result = regime_switching_logit(df_comp)
        print(result[0].summary(), result[1].summary())
        results.append(result)
    except Exception as e:
        print(f"Error processing {comp}: {e}")
        result = regime_switching_logit_single_unit(df_comp)
        print(result[0].summary(), result[1].summary())
        results.append(result)

RWE Generation
Optimization terminated successfully.
         Current function value: 0.196023
         Iterations: 71
         Function evaluations: 133
Optimization terminated successfully.
         Current function value: 0.405907
         Iterations: 66
         Function evaluations: 124
                             Logit Regression Results                            
Dep. Variable:     deviation_discretized   No. Observations:               167072
Model:                             Logit   Df Residuals:                   167070
Method:                              MLE   Df Model:                            1
Date:                   Sun, 15 Feb 2026   Pseudo R-squ.:                 0.04858
Time:                           19:37:34   Log-Likelihood:                -32750.
converged:                          True   LL-Null:                       -34422.
Covariance Type:                 cluster   LLR p-value:                     0.000
                                   coef    std err 

# Run model: slice by production type

In [17]:
techs = ['Fossil Brown coal/Lignite', 'Fossil Hard coal','Fossil Gas Combined cycle', 'Fossil Gas gas/steam turbine']

results = []
for tech in techs:
    df_tech = df.loc[df['production_type'] == tech]
    print(f'{tech}')
    result = regime_switching_logit(df_tech)
    print(result[0].summary(), result[1].summary())
    results.append(result)

Fossil Brown coal/Lignite
Optimization terminated successfully.
         Current function value: 0.070073
         Iterations: 81
         Function evaluations: 155
Optimization terminated successfully.
         Current function value: 0.689709
         Iterations: 49
         Function evaluations: 92
                             Logit Regression Results                            
Dep. Variable:     deviation_discretized   No. Observations:               180790
Model:                             Logit   Df Residuals:                   180788
Method:                              MLE   Df Model:                            1
Date:                   Sun, 15 Feb 2026   Pseudo R-squ.:                 0.03324
Time:                           19:37:38   Log-Likelihood:                -12669.
converged:                          True   LL-Null:                       -13104.
Covariance Type:                 cluster   LLR p-value:                1.845e-191
                                   coef  